### Pré-traitement

In [16]:
import pandas as pd

# On récupère les données de stats
data_ids = pd.read_csv("../data/entity_resolution.csv")
data_opta_analyst = pd.read_csv("../data/silver_analyst.csv")
data_fotmob = pd.read_csv("../data/silver_fotmob.csv")
data_sofascore = pd.read_csv("../data/silver_sofascore.csv")
data_understat = pd.read_csv("../data/silver_understat.csv")


In [17]:
import hashlib

# On crée l'identifiant issu d'opta analyst
def create_player_id(analyst_name):
    return hashlib.sha256(str(analyst_name).encode("utf-8")).hexdigest()[:16]

data_ids["opta_id"] = data_ids["analyst_name"].apply(create_player_id)

In [18]:
data_ids

,analyst_name,best_tm_id,best_method,fuzzy_score,reep_id,fotmob_id,sofascore_id,fbref_id,understat_id,opta_id
0,Aaron Greene,122407.0,reep,NaN,reep_pf7029e95,NaN,NaN,NaN,NaN,41e1a6833619b240
1,Aarón Escandell,284430.0,reep,NaN,reep_pbc6fe046,534955.0,368922.0,67669ce7,NaN,be5c61f30246a2b9
2,Aarón Herrera,401362.0,reep,NaN,reep_pcdb31119,NaN,NaN,d86e3070,NaN,bf8da63c2c534563
3,Aarón Martín,251878.0,reep,NaN,reep_pe0bb5f2e,684981.0,797286.0,2f3e911a,NaN,17b08b01696a4b12
4,Abakar Sylla,962555.0,reep,NaN,reep_peeaa665d,1359613.0,1170197.0,NaN,NaN,1b43e3b148b564e9
...,...,...,...,...,...,...,...,...,...,...
8636,Zachary Athekame,990637.0,reep,NaN,reep_p5406864a,1595629.0,1409700.0,NaN,NaN,2b627ddf55fdb08b
8637,Zander Clark,98067.0,reep,NaN,reep_p2b342b28,NaN,556366.0,d7fc839a,NaN,fce47f1f77818a44
8638,Óscar Trejo,30321.0,reep,NaN,reep_pab0700e3,21414.0,21949.0,fc647b34,NaN,5bd027a5f1bf4062
8639,Óscar Valentín,517753.0,reep,NaN,reep_p9d45ac50,956622.0,900008.0,592f3158,NaN,9486ca987c85e01a


In [19]:
# On comptabilise le nombre d'identifiants disponible pour un joueur sur les différents fournisseurs de données

cols_ids = ["best_tm_id","fotmob_id","sofascore_id","opta_id"]

n_complet = data_ids[cols_ids].notna().all(axis=1).sum()

print(n_complet)

n_total = len(data_ids)
pct_complet = n_complet / n_total * 100

print(f"{n_complet} lignes sur {n_total} ({pct_complet:.1f} %)")

1957
1957 lignes sur 8641 (22.6 %)


### Opta

In [20]:
# On ajoute l'identifiant d'opta
data_opta_analyst = data_opta_analyst.merge(
    data_ids[["analyst_name", "opta_id"]],left_on="name",
    right_on="analyst_name",how="left")

# Suppression de la colonne analyst_name ajoutée par le merge
data_opta_analyst = data_opta_analyst.drop(columns="analyst_name")

# On garde uniquement les lignes ayant opta_id
data_opta_analyst = data_opta_analyst[
    data_opta_analyst["opta_id"].notna()].copy()

### Sofascore

### Fotmob